# D51 Appendix Housing

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.10

# [51일차 실습 부록 · 프로젝트 1] 정형 데이터 예측 서비스

**이 부록은 51일차 게이트웨이 실습을 마친 뒤 52일차 통합 프로젝트에 들어가기 전에 전체 서빙 파이프라인(모델 학습 → 전처리 통계 저장 → FastAPI 서버 → Streamlit 대시보드)을 처음부터 끝까지 한 번 조립해 보는 실습입니다.**

**학습목표**  
- 캘리포니아 주택 가격 데이터를 학습 데이터 통계로 정규화하고 PyTorch 회귀 모델 가중치와 전처리 파라미터를 분리 저장해 추론 모듈(`HousingPredictor`)로 구현할 수 있다.  
- 입력 범위를 제한한 Pydantic 스키마와 `run_in_executor` 비동기 처리를 결합해 주택 가격 예측 FastAPI 서버(`POST /predict`)를 구현할 수 있다.  
- Streamlit 입력 폼에서 FastAPI 서버를 호출해 예측 결과를 표시하고 정상·에러·동시 요청 통합 테스트로 서비스 동작을 검증할 수 있다.

**중심 질문** : 학습한 모델을 API 서버와 화면으로 연결할 때 모델 가중치 외에 무엇을 함께 저장하고 맞춰야 하는가?

52일차 서빙 프로젝트 4절(선택 트랙)은 같은 데이터와 같은 조립 순서를 사이킷런 `Pipeline`으로 빠르게 반복하는 과정이고 이 부록은 PyTorch 모델을 직접 학습해 서빙합니다. 3.2절의 `app/housing_api.py`는 비동기 처리 노트 5절에서 만든 `app/logger_config.py`, `app/error_handlers.py`, `app/middleware.py`를 불러오므로 빠진 파일이 있으면 비동기 처리 노트 6.0절의 준비 셀로 먼저 생성합니다. 코드 셀 주석과 그림에 남은 "Day N"은 원본 교안의 일차 번호입니다.

## 실행 환경

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 내용 |
|---|---|
| 기준 환경 | 내 PC 가상환경 (REST API 노트 2장의 `.venv`) |
| GPU | 필요 없음 |
| 띄우는 서비스 | FastAPI 8000번, Streamlit 8501번 (원본 그대로) |
| 외부 통신 | 데이터 첫 다운로드 (figshare, 1.2절) |
| 오래 걸리는 셀 | 2.3절 학습 루프 약 10초(CPU 2개, 같은 모양의 합성 데이터로 잰 값), 서버와 Streamlit 기동 각 3초 |
| 검증한 버전 | fastapi 0.141.1, pydantic 2.13.3, uvicorn 0.53.0, streamlit 1.65.0, torch 2.14.0 CPU, scikit-learn 1.8.0 |

</div>

**코랩에서 할 때 : 코랩 런타임이 초기화되면 파일이 사라지므로 구글 드라이브의 `model-serving-course` 폴더로 이동해 시작합니다.** 내 PC에서는 이 셀을 건너뛰고 `model-serving-course` 폴더에서 노트북을 엽니다. 세션을 다시 시작해 작업 폴더가 `/content`로 돌아갔을 때도 이 셀부터 다시 실행합니다.

In [ ]:
# 코랩에서만 실행합니다. 내 PC에서는 건너뜁니다
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/model-serving-course"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("작업 폴더 :", os.getcwd())

**로컬 작업 : 코랩에서는 `localhost` 주소가 브라우저에서 바로 열리지 않으므로 Swagger UI는 `output.serve_kernel_port_as_iframe(8000, path='/docs')`로 확인하고 4장의 Streamlit 대시보드는 내 PC에서 실행합니다.**

## 1. 프로젝트 개요: 정형 데이터 기반 예측 서비스 설계

**1장에서는 정형 데이터 예측 서비스의 입출력 규격과 파일 구조를 설계합니다.**

### 1.1 오늘 만들 것

**브라우저에서 주택 정보 여덟 항목을 입력하면 FastAPI 서버와 PyTorch 모델을 거쳐 예상 주택 가격을 반환하는 서비스를 구축합니다.** REST API 노트부터 인증과 화면 노트 5장까지 MNIST 예제로 익힌 조각들을 새로운 데이터와 모델로 처음부터 끝까지 연결합니다.

```
[최종 결과물]

브라우저에서:
  1. 주택 정보(면적, 방 수, 위치 등)를 입력합니다.
  2. "가격 예측" 버튼을 누릅니다.
  3. 예상 주택 가격이 표시됩니다.

내부 구조:
  Streamlit (입력 폼) → FastAPI (추론 API) → PyTorch 모델 (가격 예측)
```

### 1.2 데이터셋: 캘리포니아 주택 가격

**정형 데이터 회귀의 대표 예제인 사이킷런의 `California Housing` 데이터셋을 사용합니다.**

In [1]:
%pip install scikit-learn

  Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 51.3 MB/s eta 0:00:00
Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl (36.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
print(f"샘플 수: {data.data.shape[0]:,}")
print(f"피처 수: {data.data.shape[1]}")
print(f"타겟: {data.target_names}")
print(f"\n피처 목록:")
for i, name in enumerate(data.feature_names):
    print(f"  {i+1}. {name}")

샘플 수: 20,640
피처 수: 8
타겟: ['MedHouseVal']

피처 목록:
  1. MedInc
  2. HouseAge
  3. AveRooms
  4. AveBedrms
  5. Population
  6. AveOccup
  7. Latitude
  8. Longitude


**더 알아보기 : `fetch_california_housing()`은 첫 실행 시 외부(figshare)에서 데이터를 받아 `~/scikit_learn_data`에 저장하므로 사내 폐쇄망에서는 반입 절차나 합성 데이터가 필요합니다.** 인터넷이 되는 PC에서 받아 둔 `scikit_learn_data` 폴더를 에어갭 RAG 노트 2절의 반입 절차로 들여오거나 서빙 프로젝트 4.2절처럼 합성 표로 흐름을 연습합니다.

**입력은 숫자 피처 여덟 개이고 출력은 중위 주택 가격(단위: $100,000)인 정형 데이터 회귀 문제입니다.** 이미지 분류였던 MNIST와 데이터 형태가 다르므로 전처리, 스키마, 화면 구성이 모두 달라집니다.

<div style="font-size: 18px; line-height: 1.6;">

| 피처 이름 | 설명 | 비고 |
|---|---|---|
| `MedInc` | 중위 소득 | 입력 피처 1 |
| `HouseAge` | 주택 연식 | 입력 피처 2 |
| `AveRooms` | 평균 방 수 | 입력 피처 3 |
| `AveBedrms` | 평균 침실 수 | 입력 피처 4 |
| `Population` | 인구 | 입력 피처 5 |
| `AveOccup` | 평균 거주 인원 | 입력 피처 6 |
| `Latitude` | 위도 | 입력 피처 7 |
| `Longitude` | 경도 | 입력 피처 8 |
| `MedHouseVal` | 중위 주택 가격 ($100,000 단위) | 예측 타깃 (샘플 수 20,640건) |

</div>

### 1.3 프로젝트 구조

**기존 MNIST 파일과 섞이지 않도록 오늘 만드는 모델, 스키마, 서버, 화면 파일에는 모두 `housing_` 접두사를 붙입니다.**

```
model-serving-course/
├── app/
│   ├── model_utils.py          ← 기존 MNIST용 (건드리지 않음)
│   ├── schemas.py              ← 기존 MNIST용 (건드리지 않음)
│   ├── housing_model.py        ← 새 파일 : 주택 가격 모델 정의 + 추론 함수
│   ├── housing_schemas.py      ← 새 파일 : 주택 가격 API 스키마
│   └── housing_api.py          ← 새 파일 : 주택 가격 FastAPI 서버
├── frontend/
│   ├── app_dashboard.py        ← 기존 MNIST용
│   └── app_housing.py          ← 새 파일 : 주택 가격 Streamlit 대시보드
├── models/
│   ├── mnist_state_dict.pth    ← 기존 MNIST용
│   └── housing_model.pth       ← 새 파일 : 주택 가격 모델 가중치
└── notebooks/
    └── day5_project1.ipynb     ← 오늘의 노트북
```

### 1.4 전체 워크플로우

**전체 작업은 모델 준비(2장) → FastAPI 백엔드(3장) → Streamlit 프론트엔드(4장) → 통합 테스트(5장) → 회고(6장) 순서로 진행합니다.**

```
[섹션 2] 모델 준비 : 데이터 로드 → 전처리 → PyTorch 모델 학습 → 저장
    ↓
[섹션 3] FastAPI 백엔드 : Pydantic 스키마 → 추론 엔드포인트 → 비동기 처리
    ↓
[섹션 4] Streamlit 프론트엔드 : 입력 폼 → API 호출 → 결과 시각화
    ↓
[섹션 5] 통합 테스트 : 정상 요청 / 에러 요청 / 동시 요청 테스트
    ↓
[섹션 6] 회고
```

각 섹션에서 앞 일차의 기술이 어떻게 조합되는지 그림으로 확인합니다.

![image.png](lms-img:0)

### 1.5 시작하기 전에

**코드 셀의 `# *your code*` 표시가 붙은 줄은 직접 작성할 수 있어야 하는 핵심 로직입니다.** 전체 코드가 함께 제공되므로 그대로 실행해 동작을 확인한 뒤 `# *your code*` 영역을 스스로 작성할 수 있는지 점검합니다.

### 체크포인트

1. 이 프로젝트의 입력과 출력은 각각 무엇입니까?
2. MNIST 프로젝트와 비교했을 때 데이터 형태가 어떻게 다릅니까?
3. 오늘 새로 만들 파일 3개의 이름과 역할을 설명해 봅니다.

## 2. 모델 준비: 학습 → 전처리 파이프라인 → 저장

**2장에서는 캘리포니아 주택 데이터를 학습 데이터 통계로 정규화해 PyTorch 회귀 모델을 학습하고 가중치와 전처리 통계를 함께 저장합니다.**

### 2.1 데이터 로드 및 탐색

**데이터셋을 불러와 피처 이름 여덟 개와 값의 분포를 확인합니다.**

In [3]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# 데이터 로드
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names

print(f"피처 크기: {X.shape}")        # (20640, 8)
print(f"타겟 크기: {y.shape}")        # (20640,)
print(f"타겟 범위: {y.min():.2f} ~ {y.max():.2f} ($100,000 단위)")
print(f"\n피처별 통계:")
for i, name in enumerate(feature_names):
    print(f"  {name:12s}  평균: {X[:, i].mean():10.2f}  표준편차: {X[:, i].std():10.2f}")

피처 크기: (20640, 8)
타겟 크기: (20640,)
타겟 범위: 0.15 ~ 5.00 ($100,000 단위)

피처별 통계:
  MedInc        평균:       3.87  표준편차:       1.90
  HouseAge      평균:      28.64  표준편차:      12.59
  AveRooms      평균:       5.43  표준편차:       2.47
  AveBedrms     평균:       1.10  표준편차:       0.47
  Population    평균:    1425.48  표준편차:    1132.43
  AveOccup      평균:       3.07  표준편차:      10.39
  Latitude      평균:      35.63  표준편차:       2.14
  Longitude     평균:    -119.57  표준편차:       2.00


In [4]:
# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # *your code* — test_size와 random_state 설정
)

print(f"학습 데이터: {X_train.shape[0]:,}개")
print(f"테스트 데이터: {X_test.shape[0]:,}개")

학습 데이터: 16,512개
테스트 데이터: 4,128개


### 2.2 전처리: 정규화

**피처마다 값의 범위가 크게 다르므로(소득 0\~15, 인구 0\~35,000) 평균을 빼고 표준편차로 나누는 정규화(Normalization)를 적용해야 모델 학습이 안정적으로 수렴합니다.**

In [5]:
# 학습 데이터의 평균/표준편차 계산 (테스트 데이터에는 학습 데이터의 통계를 사용)
train_mean = X_train.mean(axis=0)  # *your code* — 축(axis) 설정
train_std = X_train.std(axis=0)    # *your code* — 축(axis) 설정

print("피처별 평균:", np.round(train_mean, 2))
print("피처별 표준편차:", np.round(train_std, 2))

피처별 평균: [ 3.88000e+00  2.86100e+01  5.44000e+00  1.10000e+00  1.42645e+03
  3.10000e+00  3.56400e+01 -1.19580e+02]
피처별 표준편차: [1.90000e+00 1.26000e+01 2.39000e+00 4.30000e-01 1.13702e+03 1.15800e+01
 2.14000e+00 2.01000e+00]


In [6]:
# 정규화 적용
X_train_norm = (X_train - train_mean) / train_std  # *your code* — 정규화 공식
X_test_norm = (X_test - train_mean) / train_std     # 테스트에도 학습 데이터의 통계 사용

print(f"정규화 후 학습 데이터 평균: {X_train_norm.mean(axis=0).round(4)}")  # 거의 0
print(f"정규화 후 학습 데이터 표준편차: {X_train_norm.std(axis=0).round(4)}")  # 거의 1

정규화 후 학습 데이터 평균: [-0. -0.  0. -0. -0. -0.  0. -0.]
정규화 후 학습 데이터 표준편차: [1. 1. 1. 1. 1. 1. 1. 1.]


**테스트 데이터와 배포 환경의 새 데이터는 반드시 학습 데이터의 평균(`train_mean`)과 표준편차(`train_std`)로 정규화합니다.** 실제 배포 환경에서는 새로 들어올 데이터의 통계를 미리 알 수 없으므로 학습 때 구한 평균과 표준편차를 파일로 함께 저장해 추론할 때 그대로 써야 합니다.

In [7]:
# 텐서 변환
X_train_tensor = torch.FloatTensor(X_train_norm)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)  # *your code* — (N,) → (N,1)
X_test_tensor = torch.FloatTensor(X_test_norm)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

print(f"X_train 텐서: {X_train_tensor.shape}")  # torch.Size([16512, 8])
print(f"y_train 텐서: {y_train_tensor.shape}")  # torch.Size([16512, 1])

X_train 텐서: torch.Size([16512, 8])
y_train 텐서: torch.Size([16512, 1])


### 2.3 모델 정의 및 학습

**입력 8개에서 은닉층(64 → 32)을 거쳐 출력 1개를 내는 다층 퍼셉트론(`HousingNet`)을 정의하고 학습합니다.**

In [9]:
class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델 (회귀)"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),   # *your code* — 입력 차원 → 64
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),           # *your code* — 32 → 1 (회귀 출력)
        )

    def forward(self, x):
        return self.network(x)

model = HousingModel(input_dim=8)
print(f"모델 구조:\n{model}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

모델 구조:
HousingModel(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
파라미터 수: 2,689


In [10]:
# 학습 설정
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

criterion = nn.MSELoss()                                          # *your code* — 회귀이므로 MSELoss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)         # *your code* — Adam 옵티마이저

EPOCHS = 50

In [11]:
# 학습 루프
model.train()
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)          # *your code* — 순전파
        loss = criterion(predictions, y_batch) # *your code* — 손실 계산
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if epoch % 10 == 0:
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch:3d}/{EPOCHS} — Loss: {avg_loss:.4f}")

Epoch  10/50 — Loss: 0.5322
Epoch  20/50 — Loss: 0.4640
Epoch  30/50 — Loss: 0.4241
Epoch  40/50 — Loss: 0.4020
Epoch  50/50 — Loss: 0.3874


In [12]:
# 테스트 평가
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_loss = criterion(test_preds, y_test_tensor)

    # MAE (Mean Absolute Error) 계산
    mae = torch.abs(test_preds - y_test_tensor).mean().item()

print(f"테스트 MSE:  {test_loss.item():.4f}")
print(f"테스트 MAE:  {mae:.4f} ($100,000 단위)")
print(f"테스트 MAE:  ${mae * 100000:,.0f} (실제 금액)")

테스트 MSE:  0.3219
테스트 MAE:  0.3918 ($100,000 단위)
테스트 MAE:  $39,181 (실제 금액)


### 2.4 모델 및 전처리 파라미터 저장

**모델 가중치(`housing_model.pth`)와 전처리 통계(`housing_preprocessing.json`)를 각각 파일로 저장합니다.**

In [13]:
import os
os.makedirs("models", exist_ok=True)

# 모델 가중치 저장
torch.save(model.state_dict(), "models/housing_model.pth")  # *your code* — state_dict 저장
print(f"✅ 모델 저장: models/housing_model.pth ({os.path.getsize('models/housing_model.pth')/1024:.1f} KB)")

# 전처리 파라미터 저장 (배포 시 필수!)
preprocessing_params = {
    "mean": train_mean.tolist(),
    "std": train_std.tolist(),
    "feature_names": feature_names,
}

import json
with open("models/housing_preprocessing.json", "w") as f:
    json.dump(preprocessing_params, f, indent=2)

print(f"✅ 전처리 파라미터 저장: models/housing_preprocessing.json")

✅ 모델 저장: models/housing_model.pth (13.3 KB)
✅ 전처리 파라미터 저장: models/housing_preprocessing.json


**모델 가중치만 저장하고 `train_mean`, `train_std`를 저장하지 않으면 배포 환경에서 입력을 정규화할 수 없어 완전히 틀린 예측값이 나옵니다.** REST API 노트에서 `.pth` 파일만 전달하면 안 된다고 배운 것과 같은 이유입니다.

### 2.5 추론 모듈 작성

**서버 코드와 모델 코드를 분리하기 위해 모델 로드, 정규화, 추론을 담당하는 `HousingPredictor` 클래스를 `app/housing_model.py`로 작성합니다.** REST API 노트에서 `app/model_utils.py`를 분리한 것과 같은 구조입니다.

In [14]:
%%writefile app/housing_model.py
"""
Day 5 - 주택 가격 예측 모델 정의 + 추론 함수
"""
import json
import torch
import torch.nn as nn
import numpy as np


class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


class HousingPredictor:
    """모델 로드 + 전처리 + 추론을 캡슐화한 클래스"""

    def __init__(self, model_path: str, preprocessing_path: str):
        # 전처리 파라미터 로드
        with open(preprocessing_path, "r") as f:
            params = json.load(f)
        self.mean = np.array(params["mean"])
        self.std = np.array(params["std"])
        self.feature_names = params["feature_names"]

        # 모델 로드
        self.model = HousingModel(input_dim=len(self.feature_names))
        self.model.load_state_dict(
            torch.load(model_path, map_location="cpu", weights_only=True)
        )
        self.model.eval()

    def predict(self, features: dict) -> dict:
        """
        피처 딕셔너리를 받아 가격을 예측합니다.

        Args:
            features: {"MedInc": 3.5, "HouseAge": 25, ...}
        Returns:
            {"predicted_price": 2.35, "predicted_price_usd": 235000}
        """
        # 피처를 올바른 순서로 배열
        values = [features[name] for name in self.feature_names]  # *your code* — 피처 순서 맞추기

        # 정규화
        values = np.array(values, dtype=np.float32)
        normalized = (values - self.mean) / self.std              # *your code* — 정규화 적용

        # 추론
        input_tensor = torch.FloatTensor(normalized).unsqueeze(0)  # (1, 8)
        with torch.no_grad():
            output = self.model(input_tensor)

        price = output.item()
        price = max(price, 0.0)  # 음수 방지

        return {
            "predicted_price": round(price, 4),
            "predicted_price_usd": int(price * 100000),
        }

Writing app/housing_model.py


In [15]:
# 모듈 테스트
import sys
sys.path.insert(0, ".")

from app.housing_model import HousingPredictor

predictor = HousingPredictor(
    model_path="models/housing_model.pth",
    preprocessing_path="models/housing_preprocessing.json",
)

# 테스트 데이터의 첫 번째 샘플로 테스트
sample_features = {name: float(X_test[0, i]) for i, name in enumerate(feature_names)}
print(f"입력 피처: {sample_features}")

result = predictor.predict(sample_features)
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print(f"실제 가격: ${int(y_test[0] * 100000):,}")

입력 피처: {'MedInc': 1.6812, 'HouseAge': 25.0, 'AveRooms': 4.192200557103064, 'AveBedrms': 1.0222841225626742, 'Population': 1392.0, 'AveOccup': 3.8774373259052926, 'Latitude': 36.06, 'Longitude': -119.01}
예측 가격: $66,760
실제 가격: $47,700


### 체크포인트

1. 정규화에서 학습 데이터의 통계를 테스트 데이터에도 사용하는 이유는 무엇입니까?
2. 모델 가중치 외에 **함께 저장해야 하는 것**은 무엇이고 왜 필요합니까?
3. `HousingPredictor.predict()`에서 피처를 `self.feature_names` 순서로 배열하는 이유는 무엇입니까?

## 3. FastAPI 백엔드: 추론 엔드포인트 + Pydantic 스키마

**3장에서는 주택 정보 여덟 항목을 검증하는 Pydantic 스키마를 정의하고 `HousingPredictor`를 FastAPI 엔드포인트에 연결합니다.**

### 3.1 Pydantic 스키마 설계

**입력 피처 여덟 개의 타입과 허용 범위를 `app/housing_schemas.py`에 정의합니다.**

In [16]:
%%writefile app/housing_schemas.py
"""
Day 5 - 주택 가격 예측 API 스키마
"""
from pydantic import BaseModel, Field


class HousingRequest(BaseModel):
    """주택 가격 예측 요청"""
    MedInc: float = Field(..., gt=0, description="중위 소득")                        # *your code* — gt=0 설정
    HouseAge: float = Field(..., ge=0, le=100, description="주택 연식 (년)")          # *your code* — ge, le 범위
    AveRooms: float = Field(..., gt=0, description="평균 방 수")
    AveBedrms: float = Field(..., gt=0, description="평균 침실 수")
    Population: float = Field(..., gt=0, description="인구")
    AveOccup: float = Field(..., gt=0, description="평균 거주 인원")
    Latitude: float = Field(..., ge=32, le=42, description="위도 (캘리포니아 범위)")   # *your code* — 캘리포니아 위도 범위
    Longitude: float = Field(..., ge=-125, le=-114, description="경도 (캘리포니아 범위)")

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "MedInc": 3.5,
                    "HouseAge": 25.0,
                    "AveRooms": 5.0,
                    "AveBedrms": 1.0,
                    "Population": 1500.0,
                    "AveOccup": 3.0,
                    "Latitude": 37.5,
                    "Longitude": -122.0,
                }
            ]
        }
    }


class HousingResponse(BaseModel):
    """주택 가격 예측 응답"""
    success: bool = Field(description="요청 처리 성공 여부")
    predicted_price: float = Field(description="예측 가격 ($100,000 단위)")
    predicted_price_usd: int = Field(description="예측 가격 (USD)")
    input_features: dict = Field(description="입력된 피처 값")

Writing app/housing_schemas.py


**캘리포니아 지역 데이터이므로 위도(`32~42`)와 경도(`-125~-114`)에 `Field()` 범위 제한을 걸어 잘못된 입력을 422 상태 코드로 걸러냅니다.**

### 3.2 FastAPI 서버 구현

**서버 기동 시 모델을 한 번 로드하고 `POST /predict` 요청을 스레드풀(`run_in_executor`)에서 처리하는 `app/housing_api.py`를 작성합니다.**

In [17]:
%%writefile app/housing_api.py
"""
Day 5 - 주택 가격 예측 FastAPI 서버
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, HTTPException

from app.housing_schemas import HousingRequest, HousingResponse
from app.housing_model import HousingPredictor
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("housing_api")

app = FastAPI(
    title="California Housing Price API",
    description="캘리포니아 주택 가격을 예측하는 API",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

# 추론 전용 스레드풀 (Day 3에서 배운 패턴)
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="housing")  # *your code* — 스레드풀 생성

# ===== 모델 로드 =====
MODEL_PATH = "models/housing_model.pth"
PREPROCESS_PATH = "models/housing_preprocessing.json"
predictor = None


@app.on_event("startup")
async def startup():
    global predictor
    logger.info("주택 가격 모델 로드 중...")
    predictor = HousingPredictor(MODEL_PATH, PREPROCESS_PATH)  # *your code* — HousingPredictor 인스턴스 생성
    logger.info("모델 로드 완료")


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if predictor is not None else "loading",
        "model": "California Housing",
    }


@app.post("/predict", response_model=HousingResponse, tags=["Prediction"])
async def predict_housing(request: HousingRequest):
    """주택 정보를 받아 가격을 예측합니다."""
    if predictor is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    # 요청 데이터를 딕셔너리로 변환
    features = request.model_dump()  # *your code* — Pydantic 모델 → dict

    try:
        # 추론 (별도 스레드에서 실행 — Day 3 패턴)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(       # *your code* — run_in_executor 사용
            inference_executor,
            predictor.predict,
            features,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return HousingResponse(
        success=True,
        predicted_price=result["predicted_price"],
        predicted_price_usd=result["predicted_price_usd"],
        input_features=features,
    )

Writing app/housing_api.py


**더 알아보기 : 새로 작성하는 FastAPI 서버라면 `@app.on_event("startup")` 대신 서빙 프로젝트 4.4절처럼 `lifespan`으로 모델을 로드합니다.** FastAPI 0.141은 `on_event`를 폐기 예정(deprecated)으로 표시하지만 기존 코드와의 호환을 위해 지금도 정상 동작합니다.

### 3.3 서버 실행 및 Swagger 테스트

**백그라운드에서 FastAPI 서버를 기동하고 `/health`와 `/predict` 엔드포인트를 호출해 응답을 확인합니다.**

In [18]:
!pip install nest_asyncio
!pip install uvicorn


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# ⚠️ 이전 서버가 실행 중이면 커널을 재시작하세요.

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.housing_api:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨: http://localhost:8000")
print("📄 Swagger UI: http://localhost:8000/docs")

INFO:     Started server process [20980]
INFO:     Waiting for application startup.


2026-11-03 15:38:21 INFO     [housing_api] 주택 가격 모델 로드 중...
2026-11-03 15:38:21 INFO     [housing_api] 모델 로드 완료


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨: http://localhost:8000
📄 Swagger UI: http://localhost:8000/docs


In [20]:
import requests, json

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"헬스체크: {resp.json()}")

INFO:     127.0.0.1:1840 - "GET /health HTTP/1.1" 200 OK
헬스체크: {'status': 'healthy', 'model': 'California Housing'}


In [22]:
# 추론 테스트
sample_request = {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0,
}

resp = requests.post("http://localhost:8000/predict", json=sample_request)
result = resp.json()

print(f"상태 코드: {resp.status_code}")
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print(f"전체 응답:")
print(json.dumps(result, indent=2, ensure_ascii=False))

INFO:     127.0.0.1:9427 - "POST /predict HTTP/1.1" 200 OK
상태 코드: 200
예측 가격: $183,548
전체 응답:
{
  "success": true,
  "predicted_price": 1.8355,
  "predicted_price_usd": 183548,
  "input_features": {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0
  }
}


In [23]:
# 에러 테스트: 필수 필드 누락
resp = requests.post("http://localhost:8000/predict", json={"MedInc": 3.5})
print(f"필드 누락 → 상태: {resp.status_code}")

# 에러 테스트: 범위 초과
resp = requests.post("http://localhost:8000/predict", json={
    **sample_request, "Latitude": 50.0  # 캘리포니아 범위 초과
})
print(f"범위 초과 → 상태: {resp.status_code}")

POST /predict → 422 (0.0s)


INFO:     127.0.0.1:9431 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
필드 누락 → 상태: 422


POST /predict → 422 (0.001s)


INFO:     127.0.0.1:9433 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
범위 초과 → 상태: 422


### 체크포인트

1. `HousingRequest`에서 `Latitude`에 `ge=32, le=42` 제한을 넣은 이유는 무엇입니까?
2. `request.model_dump()`는 어떤 역할을 합니까?
3. `run_in_executor`를 사용하지 않으면 어떤 문제가 발생할 수 있습니까?

## 4. Streamlit 프론트엔드: 입력 폼 → API 호출 → 결과 시각화

**4장에서는 사용자가 브라우저에서 숫자 여덟 개를 입력하고 예측 가격을 확인할 수 있는 Streamlit 대시보드(`frontend/app_housing.py`)를 구현합니다.**

### 4.1 대시보드 코드 작성

**2열 레이아웃의 숫자 입력 폼(`st.number_input`)과 API 호출 함수를 갖춘 대시보드 코드를 작성합니다.**

대시보드(`frontend/app_housing.py`)에서 사용하는 주요 Streamlit 함수와 HTTP 예외 처리 메서드는 다음과 같습니다.

<div style="font-size: 18px; line-height: 1.6;">

| 함수 / 메서드 | 역할과 동작 원리 |
|---|---|
| **`st.set_page_config(page_title, layout="wide")`** | 브라우저 탭 제목과 아이콘을 지정하고 화면 전체 폭을 넓게(`wide`) 쓰도록 설정합니다(스크립트 맨 첫 줄에 호출). |
| **`st.columns(2)`** | 화면을 좌우 2개 열(`col1`, `col2`)로 나누어 8개의 주택 특성 입력 칸을 깔끔하게 나란히 배치합니다. |
| **`st.number_input(라벨, min_value, max_value, value)`** | 최소·최대 범위가 제한된 숫자 입력 위젯을 만들고 사용자가 입력한 실수(`float`) 값을 반환합니다. |
| **`resp.raise_for_status()`** | `requests` 응답 상태 코드가 `4xx`나 `5xx` 에러일 때 자동으로 `HTTPError` 예외를 발생시켜 `except` 블록에서 에러 메시지를 띄우게 합니다. |

</div>

In [24]:
%%writefile frontend/app_housing.py
"""
Day 5 - 캘리포니아 주택 가격 예측 대시보드
"""
import streamlit as st
import requests

# ===== 페이지 설정 =====
st.set_page_config(
    page_title="주택 가격 예측",
    page_icon="🏠",
    layout="wide",
)

# ===== API 호출 함수 (Day 4 패턴) =====
API_BASE = "http://localhost:8000"

def call_api(url, json_data=None, method="post"):
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()                      # *your code* — HTTP 에러 시 예외 발생
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버를 실행하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None


# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")

    health = call_api(f"{API_BASE}/health", method="get")
    if health and health.get("status") == "healthy":
        st.success("🟢 서버 연결됨")
        server_ok = True
    else:
        st.error("🔴 서버 연결 실패")
        server_ok = False

    st.divider()
    st.caption("California Housing Price Predictor")
    st.caption("Day 5 — 프로젝트 1")


# ===== 메인 영역 =====
st.title("🏠 캘리포니아 주택 가격 예측")
st.write("주택 정보를 입력하면 예상 가격을 예측합니다.")

col_input, col_result = st.columns(2)

# ----- 입력 영역 -----
with col_input:
    st.subheader("📋 주택 정보 입력")

    # 소득 & 주택 연식
    c1, c2 = st.columns(2)
    with c1:
        med_inc = st.number_input(
            "중위 소득 (MedInc)",
            min_value=0.1, max_value=20.0, value=3.5, step=0.1,   # *your code* — 범위와 기본값
        )
    with c2:
        house_age = st.number_input(
            "주택 연식 (HouseAge)",
            min_value=0.0, max_value=100.0, value=25.0, step=1.0,
        )

    # 방 수 & 침실 수
    c1, c2 = st.columns(2)
    with c1:
        ave_rooms = st.number_input(
            "평균 방 수 (AveRooms)",
            min_value=0.1, max_value=50.0, value=5.0, step=0.1,
        )
    with c2:
        ave_bedrms = st.number_input(
            "평균 침실 수 (AveBedrms)",
            min_value=0.1, max_value=20.0, value=1.0, step=0.1,
        )

    # 인구 & 거주 인원
    c1, c2 = st.columns(2)
    with c1:
        population = st.number_input(
            "인구 (Population)",
            min_value=1.0, max_value=50000.0, value=1500.0, step=100.0,
        )
    with c2:
        ave_occup = st.number_input(
            "평균 거주 인원 (AveOccup)",
            min_value=0.1, max_value=20.0, value=3.0, step=0.1,
        )

    # 위치
    c1, c2 = st.columns(2)
    with c1:
        latitude = st.number_input(
            "위도 (Latitude)",
            min_value=32.0, max_value=42.0, value=37.5, step=0.1,   # *your code* — 캘리포니아 범위
        )
    with c2:
        longitude = st.number_input(
            "경도 (Longitude)",
            min_value=-125.0, max_value=-114.0, value=-122.0, step=0.1,
        )


# ----- 결과 영역 -----
with col_result:
    st.subheader("📊 예측 결과")

    if not server_ok:
        st.error("서버에 연결할 수 없습니다.")
    else:
        if st.button("🚀 가격 예측", type="primary", use_container_width=True):
            # 요청 데이터 구성
            request_data = {                            # *your code* — 입력값을 dict로 구성
                "MedInc": med_inc,
                "HouseAge": house_age,
                "AveRooms": ave_rooms,
                "AveBedrms": ave_bedrms,
                "Population": population,
                "AveOccup": ave_occup,
                "Latitude": latitude,
                "Longitude": longitude,
            }

            with st.spinner("예측 중..."):
                result = call_api(f"{API_BASE}/predict", json_data=request_data)

            if result:
                st.session_state["last_housing_result"] = result

        # 결과 표시
        if "last_housing_result" in st.session_state:
            result = st.session_state["last_housing_result"]

            # 가격 메트릭
            st.metric(
                label="예상 주택 가격",
                value=f"${result['predicted_price_usd']:,}",
            )

            st.caption(f"모델 출력값: {result['predicted_price']} ($100,000 단위)")

            # 입력 피처 확인
            with st.expander("📋 입력된 피처 확인"):
                for key, value in result["input_features"].items():
                    st.write(f"**{key}**: {value}")

Writing frontend/app_housing.py


### 4.2 실행 및 테스트

**백엔드 서버가 켜진 상태에서 Streamlit 대시보드를 실행해 화면 동작을 확인합니다.**

아래 명령어에서 **`uvicorn app.housing_api:app --port 8000`**은 `app/housing_api.py` 파일 안의 `app` 객체를 `8000`번 포트에서 백엔드 서버로 띄우는 명령어이고, **`streamlit run frontend/app_housing.py --server.port 8501`**은 작성한 대시보드 스크립트를 `8501`번 포트에서 웹 화면으로 실행하는 명령어입니다.

In [25]:
# 터미널 1: 백엔드
!uvicorn app.housing_api:app --port 8000

# 터미널 2: 프론트엔드
!streamlit run frontend/app_housing.py --server.port 8501

^C
^C


**로컬 작업 : 주피터 노트북 안에서는 `subprocess.Popen`으로 기동하는 셀만 실행하고 `streamlit run` 명령은 별도 터미널에서 실행합니다.** 노트북 코드 셀에서 `streamlit run`을 직접 실행하면 프로세스가 종료되지 않아 셀이 멈춘 채로 남습니다.

In [ ]:
# 또는 노트북에서 백그라운드 실행
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_housing.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("✅ 프론트엔드: http://localhost:8501")

![image.png](lms-img:0)

#### 테스트 시나리오

```
[테스트 1] 기본 흐름
  1. 사이드바 서버 상태 초록 확인
  2. 기본값 그대로 "가격 예측" 클릭
  3. 예상 가격이 표시되는지 확인

[테스트 2] 값 변경
  1. 중위 소득을 8.0으로 올리기 → 가격 예측 → 가격 상승 확인
  2. 중위 소득을 1.0으로 내리기 → 가격 예측 → 가격 하락 확인

[테스트 3] 에러 상황
  1. 백엔드 서버 종료 → 사이드바 빨강 확인
```

### 체크포인트

1. MNIST 대시보드(인증과 화면 노트)와 비교했을 때 입력 방식이 어떻게 다릅니까?
2. `st.number_input()`에서 `min_value`, `max_value`를 설정하는 이유는 무엇입니까?
3. `request_data` 딕셔너리의 키 이름이 `HousingRequest` 스키마의 필드 이름과 정확히 일치해야 하는 이유는 무엇입니까?

## 5. 통합 테스트 및 디버깅

**5장에서는 정상 요청, 범위 밖 에러 요청(422), 동시 요청(10건), 헬스체크의 네 가지 시나리오로 서비스를 통합 테스트합니다.**

### 5.1 API 레벨 통합 테스트

In [26]:
# ⚠️ FastAPI 서버가 실행 중이어야 합니다.

import requests
import json
import time

API_BASE = "http://localhost:8000"

print("=" * 60)
print("  통합 테스트")
print("=" * 60)

  통합 테스트


#### 테스트 1: 정상 요청

In [27]:
# 다양한 입력으로 모델 동작 확인
test_cases = [
    {"name": "저소득 지역", "MedInc": 1.5, "HouseAge": 40, "AveRooms": 4.0, "AveBedrms": 1.0,
     "Population": 2000, "AveOccup": 3.5, "Latitude": 34.0, "Longitude": -118.0},
    {"name": "고소득 지역", "MedInc": 10.0, "HouseAge": 10, "AveRooms": 8.0, "AveBedrms": 2.0,
     "Population": 500, "AveOccup": 2.0, "Latitude": 37.8, "Longitude": -122.4},
    {"name": "평균적 주택", "MedInc": 3.5, "HouseAge": 25, "AveRooms": 5.0, "AveBedrms": 1.0,
     "Population": 1500, "AveOccup": 3.0, "Latitude": 37.5, "Longitude": -122.0},
]

print("\n[테스트 1] 정상 요청 — 다양한 입력")
print(f"{'케이스':<15} {'예측 가격':>12}")
print("-" * 30)

for case in test_cases:
    name = case.pop("name")
    resp = requests.post(f"{API_BASE}/predict", json=case)    # *your code* — POST 요청
    result = resp.json()
    print(f"{name:<15} ${result['predicted_price_usd']:>10,}")
    case["name"] = name  # 복원


[테스트 1] 정상 요청 — 다양한 입력
케이스                    예측 가격
------------------------------
저소득 지역          $   102,686
고소득 지역          $   474,221
평균적 주택          $   183,548


#### 테스트 2: 에러 상황

In [28]:
print("\n[테스트 2] 에러 상황")

# 필수 필드 누락
resp = requests.post(f"{API_BASE}/predict", json={"MedInc": 3.5})
print(f"  필드 누락      → HTTP {resp.status_code}")

# 범위 초과 (위도)
bad_request = {
    "MedInc": 3.5, "HouseAge": 25, "AveRooms": 5, "AveBedrms": 1,
    "Population": 1500, "AveOccup": 3, "Latitude": 50.0, "Longitude": -122.0,  # 위도 초과
}
resp = requests.post(f"{API_BASE}/predict", json=bad_request)
print(f"  위도 범위 초과  → HTTP {resp.status_code}")

# 음수 값
bad_request2 = {
    "MedInc": -1.0, "HouseAge": 25, "AveRooms": 5, "AveBedrms": 1,
    "Population": 1500, "AveOccup": 3, "Latitude": 37.5, "Longitude": -122.0,
}
resp = requests.post(f"{API_BASE}/predict", json=bad_request2)
print(f"  소득 음수      → HTTP {resp.status_code}")

# JSON이 아닌 요청
resp = requests.post(f"{API_BASE}/predict", data="not json")
print(f"  잘못된 포맷    → HTTP {resp.status_code}")


[테스트 2] 에러 상황
  필드 누락      → HTTP 422
  위도 범위 초과  → HTTP 422
  소득 음수      → HTTP 422
  잘못된 포맷    → HTTP 422


#### 테스트 3: 동시 요청

In [29]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_predict(i):
    case = test_cases[i % len(test_cases)].copy()
    case.pop("name", None)
    start = time.time()
    resp = requests.post(f"{API_BASE}/predict", json=case, timeout=30)
    return {"id": i+1, "elapsed": round(time.time() - start, 3), "status": resp.status_code}

print("\n[테스트 3] 동시 요청 (8개)")
start = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(send_predict, i) for i in range(8)]
    results = [f.result() for f in as_completed(futures)]

total = round(time.time() - start, 2)
for r in sorted(results, key=lambda x: x["id"]):
    print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
print(f"  전체: {total}초")


[테스트 3] 동시 요청 (8개)
  요청 #1: 2.037초 (HTTP 200)
  요청 #2: 2.072초 (HTTP 200)
  요청 #3: 2.062초 (HTTP 200)
  요청 #4: 2.058초 (HTTP 200)
  요청 #5: 2.062초 (HTTP 200)
  요청 #6: 2.061초 (HTTP 200)
  요청 #7: 2.058초 (HTTP 200)
  요청 #8: 2.055초 (HTTP 200)
  전체: 2.09초


#### 테스트 4: 헬스체크

In [30]:
print("\n[테스트 4] 헬스체크")
resp = requests.get(f"{API_BASE}/health")
print(f"  상태: {resp.json()}")


[테스트 4] 헬스체크
  상태: {'status': 'healthy', 'model': 'California Housing'}


### 5.2 테스트 결과 종합

**네 가지 시나리오를 한 번에 실행해 전체 통과 여부를 요약 출력합니다.**

In [31]:
print("\n" + "=" * 60)
print("  테스트 결과 종합")
print("=" * 60)
print("  ✅ 정상 요청: 다양한 입력에서 합리적인 가격 반환")
print("  ✅ 에러 처리: 잘못된 입력에 422/400 반환, 서버 안 죽음")
print("  ✅ 동시 처리: 8개 동시 요청 정상 처리")
print("  ✅ 헬스체크: 서버 상태 정상")


  테스트 결과 종합
  ✅ 정상 요청: 다양한 입력에서 합리적인 가격 반환
  ✅ 에러 처리: 잘못된 입력에 422/400 반환, 서버 안 죽음
  ✅ 동시 처리: 8개 동시 요청 정상 처리
  ✅ 헬스체크: 서버 상태 정상


GET / → 404 (0.001s)


INFO:     77.239.124.175:49330 - "GET / HTTP/1.1" 404 Not Found


GET / → 404 (0.002s)


INFO:     199.165.159.41:37120 - "GET / HTTP/1.1" 404 Not Found


GET /favicon.ico → 404 (0.0s)


INFO:     199.165.159.47:31195 - "GET /favicon.ico HTTP/1.1" 404 Not Found


GET / → 404 (0.0s)


INFO:     199.165.159.44:54403 - "GET /?format=json HTTP/1.1" 404 Not Found


## 6. 회고: 무엇이 어려웠고, 어떻게 개선할 수 있는가?

## 6. 회고: 무엇이 어려웠고 어떻게 개선할 수 있는가?

### 6.1 오늘 만든 것 정리

**데이터 정규화부터 PyTorch 모델 학습, FastAPI 추론 서버, Streamlit 대시보드, 통합 테스트까지 정형 데이터 예측 서비스 한 벌을 완성했습니다.**

![image.png](lms-img:0)

### 6.2 REST API 노트\~인증과 화면 노트 5장과의 연결

![image.png](lms-img:0)

### 6.3 개선할 수 있는 점

![image.png](lms-img:0)

**더 알아보기 : 그림 속 "Day 8에서 컨테이너화"는 이 과정에서 50일차 Docker 컨테이너화 노트가 맡았습니다.**

**이 개선점들은 52일차 통합 프로젝트에서 직접 구현하거나 이후 MLOps 과정에서 심화합니다.**

## 최종 체크포인트

1. **전처리 통계 동반 저장** : 전처리 파라미터(`mean`, `std`)를 모델과 함께 저장해야 하는 이유는 무엇입니까?
2. **스키마 범위 검증** : `HousingRequest`에서 `Latitude`에 `ge=32, le=42`를 넣은 이유는 무엇입니까?
3. **필드 이름 일치** : Streamlit의 입력값 이름이 Pydantic 스키마의 필드 이름과 일치해야 하는 이유는 무엇입니까?
4. **비동기 스레드 분리** : 이 프로젝트에서 `run_in_executor`를 제거하면 어떤 문제가 생길 수 있습니까?
5. **태스크와 파이프라인 차이** : MNIST 프로젝트와 오늘 프로젝트의 가장 큰 차이는 무엇입니까?

<details class="lms-answer">
<summary>풀이 보기</summary>

1. **전처리 통계 동반 저장** : 모델이 학습 데이터의 평균과 표준편차로 정규화된 입력 위에서 학습되었으므로 추론할 때도 동일한 통계값으로 정규화해야 올바른 예측값이 나오기 때문입니다.  
2. **스키마 범위 검증** : 캘리포니아의 실제 위도 범위(32~42도)를 벗어난 잘못된 입력을 모델 추론 전에 `422 Unprocessable Entity`로 차단하기 위해서입니다.  
3. **필드 이름 일치** : FastAPI의 Pydantic 모델이 JSON 본문의 키 이름을 스키마 필드 이름과 대조해 파싱하므로 이름이 다르면 필수 필드 누락으로 `422` 에러가 발생합니다.  
4. **비동기 스레드 분리** : `async def` 엔드포인트 안에서 동기 함수인 PyTorch 추론을 직접 호출하면 추론이 끝날 때까지 이벤트 루프가 블로킹되어 동시 요청과 헬스체크 응답이 지연됩니다.  
5. **태스크와 파이프라인 차이** : MNIST는 28×28 이미지를 받아 0~9 클래스를 분류하는 이미지 분류였고 이번 프로젝트는 8개 수치형 피처를 정규화해 연속적인 주택 가격을 예측하는 정형 데이터 회귀입니다.

</details>

### 오늘 정리

**정형 데이터 예측 서비스는 학습 데이터의 전처리 통계, Pydantic 스키마의 필드 이름과 범위, 추론 모듈의 피처 순서가 모두 일치해야 올바른 예측값을 반환합니다.**

- **모델 준비(2장)** : 캘리포니아 주택 가격 데이터를 학습 데이터 통계로 정규화하고 PyTorch 회귀 모델 가중치(`housing_model.pth`)와 전처리 파라미터(`housing_preprocessing.json`)를 함께 저장했습니다.
- **백엔드와 화면(3~4장)** : 위도·경도 범위를 제한한 Pydantic 스키마와 FastAPI 서버를 만들고 Streamlit 입력 폼 대시보드와 연결했습니다.
- **통합 테스트(5장)** : 정상 요청(200), 범위 밖 입력(422), 동시 10건 요청, 헬스체크를 코드로 검증했습니다.
- **다음 단계(52일차 프로젝트)** : 오늘 익힌 단일 서비스 조립 순서에 API Key 인증과 비정형 데이터(텍스트 생성·멀티턴 대화) 처리를 더해 트랜스포머 챗봇 서비스와 자율 모델 서빙 프로젝트로 확장합니다.

**다음 프로젝트 연결** : 오늘 만든 정형 데이터 예측 서비스는 누구나 호출할 수 있는 수치 입력 기반의 기본 서버였습니다. 다음 단계인 `[프로젝트 2] 트랜스포머 챗봇 서비스`와 `자율 프로젝트`에서는 여기에 API Key 인증을 결합하고 정형 데이터를 넘어 텍스트·이미지 같은 비정형 데이터 모델을 서빙합니다.

### 제출

다음 내역을 마크다운 파일로 정리해 사내 저장소에 업로드하고 링크로 제출합니다.

1. 통합 테스트(5.1~5.2절) 실행 결과와 화면 캡처
2. 각 절 체크포인트의 답변
3. 프로젝트 회고

실습이 끝난 뒤에는 백그라운드에서 실행 중인 FastAPI 서버와 Streamlit 프로세스를 종료합니다.

## Reference

보충 셀의 버전과 동작은 2026년 10월 7일에 이 부록의 검증 환경에서 직접 돌려 확인했습니다. 문서 링크도 같은 날 기준입니다.

**데이터**  
출처 : scikit-learn, [Real world datasets : California Housing dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset), 2026.10.07 확인 (큰 데이터는 필요할 때 다운로드한다는 설명)  
출처 : scikit-learn 1.8.0 소스 `sklearn/datasets/_california_housing.py`, 2026.10.07 확인 (다운로드하는 주소 `https://ndownloader.figshare.com/files/5976036`)

**서버**  
출처 : FastAPI, [Lifespan Events](https://fastapi.tiangolo.com/advanced/events/), 2026.10.07 확인  
출처 : FastAPI 0.141.1 소스 `fastapi/applications.py`, 2026.10.07 확인 (`on_event`에 붙은 `@deprecated` 표시)